# Sprint 2 - Pipeline ETL de DRE por PDV

Este notebook realiza a leitura dos dados sint?ticos, valida a qualidade, consolida a base anal?tica por `id_pdv` e `mes`, calcula m?tricas de rentabilidade e cria flags de alerta para monitoramento.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = DATA_DIR / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('DATA_DIR existe:', DATA_DIR.exists())


## 1) Leitura dos CSVs


In [ ]:
cadastro = pd.read_csv(DATA_DIR / 'cadastro_pdvs.csv', parse_dates=['data_abertura'])
vendas = pd.read_csv(DATA_DIR / 'vendas_pdv_mensal.csv', parse_dates=['mes'])
custos_var = pd.read_csv(DATA_DIR / 'custos_variaveis_pdv_mensal.csv', parse_dates=['mes'])
custos_fix = pd.read_csv(DATA_DIR / 'custos_fixos_pdv_mensal.csv', parse_dates=['mes'])
metas = pd.read_csv(DATA_DIR / 'metas_orcamento_pdv_mensal.csv', parse_dates=['mes'])

for nome, df in {
    'cadastro': cadastro,
    'vendas': vendas,
    'custos_var': custos_var,
    'custos_fix': custos_fix,
    'metas': metas,
}.items():
    print(f'{nome:<12} -> {df.shape[0]:>5} linhas | {df.shape[1]:>2} colunas')


## 2) Limpeza e padroniza??o


In [ ]:
def padronizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    if 'id_pdv' in out.columns:
        out['id_pdv'] = out['id_pdv'].astype(str).str.strip().str.upper()
    return out

cadastro = padronizar_colunas(cadastro)
vendas = padronizar_colunas(vendas)
custos_var = padronizar_colunas(custos_var)
custos_fix = padronizar_colunas(custos_fix)
metas = padronizar_colunas(metas)

tabelas_mensais = [vendas, custos_var, custos_fix, metas]
for df in tabelas_mensais:
    df['mes'] = pd.to_datetime(df['mes']).dt.to_period('M').dt.to_timestamp()

numeric_cols = {
    'vendas': ['receita_bruta', 'receita_liquida', 'devolucoes'],
    'custos_var': ['cmv', 'impostos', 'comissoes'],
    'custos_fix': ['aluguel', 'folha', 'energia', 'tecnologia'],
    'metas': ['meta_receita', 'meta_margem', 'orcamento_mensal'],
}

for nome, cols in numeric_cols.items():
    df = {'vendas': vendas, 'custos_var': custos_var, 'custos_fix': custos_fix, 'metas': metas}[nome]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('Padronizacao concluida.')


## 3) Valida??es de qualidade


In [ ]:
def validar_estrutura() -> None:
    assert cadastro['id_pdv'].is_unique, 'Cadastro possui id_pdv duplicado'

    chaves = ['id_pdv', 'mes']
    for nome, df in {
        'vendas': vendas,
        'custos_var': custos_var,
        'custos_fix': custos_fix,
        'metas': metas,
    }.items():
        dup = df.duplicated(chaves).sum()
        assert dup == 0, f'{nome} possui duplicidade de chave: {dup}'

    key_vendas = set(map(tuple, vendas[chaves].to_records(index=False)))
    for nome, df in {'custos_var': custos_var, 'custos_fix': custos_fix, 'metas': metas}.items():
        key_df = set(map(tuple, df[chaves].to_records(index=False)))
        assert key_df == key_vendas, f'Chaves diferentes entre vendas e {nome}'

    assert (vendas['receita_bruta'] >= vendas['receita_liquida']).all(), 'Receita liquida maior que receita bruta'
    assert (vendas['devolucoes'] >= 0).all(), 'Devolucoes negativas'
    for col in ['cmv', 'impostos', 'comissoes']:
        assert (custos_var[col] >= 0).all(), f'{col} negativo'
    for col in ['aluguel', 'folha', 'energia', 'tecnologia']:
        assert (custos_fix[col] >= 0).all(), f'{col} negativo'

validar_estrutura()
print('Validacoes concluidas sem erros.')


## 4) Consolida??o da base anal?tica por PDV e m?s


In [ ]:
dre = (
    vendas
    .merge(custos_var, on=['id_pdv', 'mes'], how='left')
    .merge(custos_fix, on=['id_pdv', 'mes'], how='left')
    .merge(metas, on=['id_pdv', 'mes'], how='left')
    .merge(cadastro, on='id_pdv', how='left')
)

for col in ['cmv', 'impostos', 'comissoes', 'aluguel', 'folha', 'energia', 'tecnologia', 'meta_receita', 'meta_margem', 'orcamento_mensal']:
    dre[col] = dre[col].fillna(0)

dre = dre.sort_values(['id_pdv', 'mes']).reset_index(drop=True)
print('Base consolidada:', dre.shape)
dre.head(3)


## 5) C?lculo de m?tricas de rentabilidade


In [ ]:
dre['lucro_bruto'] = dre['receita_liquida'] - dre['cmv']
dre['despesas_variaveis'] = dre['impostos'] + dre['comissoes']
dre['despesas_fixas'] = dre['aluguel'] + dre['folha'] + dre['energia'] + dre['tecnologia']

dre['depreciacao_amortizacao'] = dre['despesas_fixas'] * 0.03
dre['resultado_financeiro'] = -0.015 * dre['receita_liquida']

dre['ebitda'] = dre['lucro_bruto'] - dre['despesas_variaveis'] - dre['despesas_fixas']
dre['lucro_operacional'] = dre['ebitda'] - dre['depreciacao_amortizacao']
base_imposto = dre['lucro_operacional'] + dre['resultado_financeiro']
dre['ir_csll'] = np.where(base_imposto > 0, base_imposto * 0.24, 0.0)
dre['lucro_liquido'] = base_imposto - dre['ir_csll']

dre['margem_bruta'] = np.where(dre['receita_liquida'] > 0, dre['lucro_bruto'] / dre['receita_liquida'], np.nan)
dre['margem_operacional'] = np.where(dre['receita_liquida'] > 0, dre['lucro_operacional'] / dre['receita_liquida'], np.nan)
dre['margem_ebitda'] = np.where(dre['receita_liquida'] > 0, dre['ebitda'] / dre['receita_liquida'], np.nan)
dre['margem_liquida'] = np.where(dre['receita_liquida'] > 0, dre['lucro_liquido'] / dre['receita_liquida'], np.nan)

cols_metricas = [
    'id_pdv', 'mes', 'receita_liquida', 'lucro_bruto', 'ebitda', 'lucro_liquido',
    'margem_bruta', 'margem_operacional', 'margem_ebitda', 'margem_liquida'
]
dre[cols_metricas].head(5)


## 6) Flags de alerta


In [ ]:
dre['receita_liquida_mes_anterior'] = dre.groupby('id_pdv')['receita_liquida'].shift(1)
dre['var_receita_vs_mes_anterior'] = np.where(
    dre['receita_liquida_mes_anterior'] > 0,
    (dre['receita_liquida'] / dre['receita_liquida_mes_anterior']) - 1,
    np.nan
)

dre['flag_margem_abaixo_10'] = dre['margem_operacional'] < 0.10
dre['flag_queda_receita_20'] = dre['var_receita_vs_mes_anterior'] <= -0.20
dre['flag_ebitda_negativo'] = dre['ebitda'] < 0

dre['score_alerta'] = (
    dre['flag_margem_abaixo_10'].astype(int)
    + dre['flag_queda_receita_20'].astype(int)
    + dre['flag_ebitda_negativo'].astype(int)
)

dre[['flag_margem_abaixo_10', 'flag_queda_receita_20', 'flag_ebitda_negativo']].mean().rename('percentual_alerta')


## 7) Export da base consolidada


In [ ]:
output_path = OUTPUT_DIR / 'dre_pdv_mensal_consolidado.csv'
dre.to_csv(output_path, index=False, encoding='utf-8')
print('Arquivo salvo em:', output_path)
print('Linhas:', len(dre), '| Colunas:', len(dre.columns))


## Sprint 3 - Camada de Insights e Diagn?stico


### 9) Top 10 PDVs menos rent?veis


In [ ]:
resumo_pdv = (
    dre.groupby(['id_pdv', 'regiao', 'canal'], as_index=False)
    .agg(
        receita_liquida_media=('receita_liquida', 'mean'),
        receita_liquida_total=('receita_liquida', 'sum'),
        ebitda_total=('ebitda', 'sum'),
        lucro_liquido_total=('lucro_liquido', 'sum'),
        margem_operacional_media=('margem_operacional', 'mean'),
        margem_ebitda_media=('margem_ebitda', 'mean'),
        devolucao_media_pct=('devolucoes', lambda s: s.sum() / max(dre.loc[s.index, 'receita_bruta'].sum(), 1.0)),
        custo_fixo_medio_pct=('despesas_fixas', lambda s: s.sum() / max(dre.loc[s.index, 'receita_liquida'].sum(), 1.0)),
    )
)

top10_menos_rentaveis = (
    resumo_pdv
    .sort_values(['lucro_liquido_total', 'margem_operacional_media'], ascending=[True, True])
    .head(10)
    .reset_index(drop=True)
)

top10_menos_rentaveis


### 10) Principais drivers de queda de rentabilidade


In [ ]:
driver_base = resumo_pdv.copy()

p75_custo_fixo = driver_base['custo_fixo_medio_pct'].quantile(0.75)
p75_devolucao = driver_base['devolucao_media_pct'].quantile(0.75)
p25_receita = max(driver_base['receita_liquida_media'].quantile(0.25), 1.0)

driver_base['flag_fixo_alto'] = driver_base['custo_fixo_medio_pct'] >= p75_custo_fixo
driver_base['flag_devolucao_alta'] = driver_base['devolucao_media_pct'] >= p75_devolucao
# Proxy para ticket baixo: PDV com receita media recorrente no quartil inferior
driver_base['flag_ticket_baixo'] = driver_base['receita_liquida_media'] <= p25_receita

driver_base['score_fixo'] = driver_base['custo_fixo_medio_pct'] / max(p75_custo_fixo, 1e-6)
driver_base['score_devolucao'] = driver_base['devolucao_media_pct'] / max(p75_devolucao, 1e-6)
driver_base['score_ticket'] = p25_receita / np.maximum(driver_base['receita_liquida_media'], 1.0)

score_cols = ['score_fixo', 'score_devolucao', 'score_ticket']
driver_map = {
    'score_fixo': 'fixo_alto',
    'score_devolucao': 'devolucoes_altas',
    'score_ticket': 'ticket_baixo',
}

driver_base['driver_principal'] = (
    driver_base[score_cols]
    .idxmax(axis=1)
    .map(driver_map)
)

drivers_pdv = driver_base[[
    'id_pdv', 'regiao', 'canal',
    'custo_fixo_medio_pct', 'devolucao_media_pct', 'receita_liquida_media',
    'flag_fixo_alto', 'flag_devolucao_alta', 'flag_ticket_baixo',
    'driver_principal'
]].sort_values(['driver_principal', 'id_pdv']).reset_index(drop=True)

drivers_pdv.head(15)


### 11) Compara??es regionais


In [ ]:
comparativo_regional = (
    dre.groupby(['regiao', 'mes'], as_index=False)
    .agg(
        receita_liquida=('receita_liquida', 'sum'),
        ebitda=('ebitda', 'sum'),
        lucro_liquido=('lucro_liquido', 'sum'),
        receita_bruta=('receita_bruta', 'sum'),
        devolucoes=('devolucoes', 'sum'),
        despesas_fixas=('despesas_fixas', 'sum'),
    )
)

comparativo_regional['margem_ebitda'] = np.where(
    comparativo_regional['receita_liquida'] > 0,
    comparativo_regional['ebitda'] / comparativo_regional['receita_liquida'],
    np.nan,
)
comparativo_regional['margem_liquida'] = np.where(
    comparativo_regional['receita_liquida'] > 0,
    comparativo_regional['lucro_liquido'] / comparativo_regional['receita_liquida'],
    np.nan,
)
comparativo_regional['pct_devolucao'] = np.where(
    comparativo_regional['receita_bruta'] > 0,
    comparativo_regional['devolucoes'] / comparativo_regional['receita_bruta'],
    np.nan,
)
comparativo_regional['pct_custo_fixo'] = np.where(
    comparativo_regional['receita_liquida'] > 0,
    comparativo_regional['despesas_fixas'] / comparativo_regional['receita_liquida'],
    np.nan,
)

resumo_regional_geral = (
    comparativo_regional.groupby('regiao', as_index=False)
    .agg(
        receita_liquida_total=('receita_liquida', 'sum'),
        ebitda_total=('ebitda', 'sum'),
        lucro_liquido_total=('lucro_liquido', 'sum'),
        margem_ebitda_media=('margem_ebitda', 'mean'),
        margem_liquida_media=('margem_liquida', 'mean'),
        devolucao_media_pct=('pct_devolucao', 'mean'),
        custo_fixo_medio_pct=('pct_custo_fixo', 'mean'),
    )
    .sort_values('margem_liquida_media', ascending=False)
    .reset_index(drop=True)
)

resumo_regional_geral


### 12) Export dos artefatos de insights


In [ ]:
top10_path = OUTPUT_DIR / 'insights_top10_pdvs_menos_rentaveis.csv'
drivers_path = OUTPUT_DIR / 'insights_drivers_pdv.csv'
regional_path = OUTPUT_DIR / 'insights_comparativo_regional.csv'

top10_menos_rentaveis.to_csv(top10_path, index=False, encoding='utf-8')
drivers_pdv.to_csv(drivers_path, index=False, encoding='utf-8')
comparativo_regional.to_csv(regional_path, index=False, encoding='utf-8')

print('Arquivos gerados:')
print('-', top10_path)
print('-', drivers_path)
print('-', regional_path)


## 13) Resumo executivo do Sprint 3

- Ranking dos 10 PDVs com menor rentabilidade consolidada.
- Diagn?stico de driver principal por PDV (`fixo_alto`, `ticket_baixo`, `devolucoes_altas`).
- Comparativo regional mensal com margens e indicadores de press?o operacional.


## Sprint 4 - Planos de A??o Recomendados


### 14) Recomenda??o por PDV com base no driver principal


In [ ]:
risco_pdv = (
    dre.groupby('id_pdv', as_index=False)
    .agg(
        score_alerta_medio=('score_alerta', 'mean'),
        meses_criticos=('score_alerta', lambda s: int((s >= 2).sum())),
        margem_operacional_media=('margem_operacional', 'mean'),
        pct_meses_queda_receita=('flag_queda_receita_20', 'mean'),
        pct_meses_ebitda_negativo=('flag_ebitda_negativo', 'mean'),
    )
)

recomendacao_base = drivers_pdv.merge(risco_pdv, on='id_pdv', how='left')

recomendacao_base['prioridade_risco'] = np.select(
    [
        (recomendacao_base['meses_criticos'] >= 6) | (recomendacao_base['margem_operacional_media'] < 0),
        (recomendacao_base['meses_criticos'] >= 3) | (recomendacao_base['margem_operacional_media'] < 0.10),
    ],
    ['alta', 'media'],
    default='baixa',
)

acao_catalogo = {
    'fixo_alto': [
        ('Reduzir custo fixo', 'Renegociar aluguel e contratos de servicos para reduzir em ao menos 8% os gastos fixos em ate 90 dias.', 90),
        ('Otimizacao de escala', 'Readequar escala da equipe por faixas de demanda para elevar produtividade sem perda de atendimento.', 60),
        ('Eficiencia operacional', 'Implementar plano de eficiencia em energia e tecnologia para cortar desperdicios recorrentes.', 60),
    ],
    'ticket_baixo': [
        ('Revisar mix de produtos', 'Ajustar mix e precificacao local para aumentar ticket medio e margem de contribuicao.', 60),
        ('Acao de marketing local', 'Ativar campanha local de upsell e cross-sell com foco em categorias de maior margem.', 45),
        ('Reforcar treinamento de equipe', 'Treinar equipe comercial para elevar conversao e venda consultiva no PDV.', 30),
    ],
    'devolucoes_altas': [
        ('Reduzir devolucoes', 'Auditar causas de devolucao por SKU e fornecedor e corrigir os principais ofensores.', 45),
        ('Padronizar processo de venda', 'Aplicar checklist de venda/troca para reduzir erro operacional e divergencia de expectativa.', 30),
        ('Qualidade e atendimento', 'Treinar equipe para orientar melhor o cliente e reduzir devolucoes por insatisfacao.', 30),
    ],
}

def quantidade_recomendacoes(prioridade: str) -> int:
    if prioridade == 'alta':
        return 3
    if prioridade == 'media':
        return 2
    return 1

linhas_recomendacao = []
for row in recomendacao_base.itertuples(index=False):
    acoes = acao_catalogo.get(row.driver_principal, [])
    qtd = quantidade_recomendacoes(row.prioridade_risco)
    for ordem, (categoria, acao, prazo) in enumerate(acoes[:qtd], start=1):
        linhas_recomendacao.append({
            'id_pdv': row.id_pdv,
            'driver_principal': row.driver_principal,
            'prioridade_risco': row.prioridade_risco,
            'recomendacao_ordem': ordem,
            'categoria_acao': categoria,
            'recomendacao': acao,
            'prazo_dias': prazo,
        })

plano_acao_pdv = pd.DataFrame(linhas_recomendacao).sort_values(['id_pdv', 'recomendacao_ordem']).reset_index(drop=True)

recomendacoes_resumo = (
    plano_acao_pdv.groupby('id_pdv', as_index=False)
    .agg(
        qtd_recomendacoes=('recomendacao_ordem', 'max'),
        recomendacoes=('recomendacao', lambda s: ' | '.join(s)),
    )
    .merge(
        recomendacao_base[['id_pdv', 'driver_principal', 'prioridade_risco']].drop_duplicates(),
        on='id_pdv',
        how='left',
    )
)

plano_acao_pdv.head(12)


### 15) Vincular recomenda??es ? base principal


In [ ]:
dre_com_recomendacoes = dre.merge(
    recomendacoes_resumo[['id_pdv', 'driver_principal', 'prioridade_risco', 'qtd_recomendacoes', 'recomendacoes']],
    on='id_pdv',
    how='left',
)

dre_com_recomendacoes[['id_pdv', 'mes', 'receita_liquida', 'margem_operacional', 'driver_principal', 'prioridade_risco', 'qtd_recomendacoes']].head(10)


### 16) Export dos artefatos de recomenda??es


In [ ]:
plano_acao_path = OUTPUT_DIR / 'plano_acao_recomendado_pdv.csv'
plano_acao_resumo_path = OUTPUT_DIR / 'plano_acao_resumo_pdv.csv'
dre_com_acoes_path = OUTPUT_DIR / 'dre_pdv_mensal_consolidado_com_acoes.csv'

plano_acao_pdv.to_csv(plano_acao_path, index=False, encoding='utf-8')
recomendacoes_resumo.to_csv(plano_acao_resumo_path, index=False, encoding='utf-8')
dre_com_recomendacoes.to_csv(dre_com_acoes_path, index=False, encoding='utf-8')

print('Arquivos gerados:')
print('-', plano_acao_path)
print('-', plano_acao_resumo_path)
print('-', dre_com_acoes_path)


## 17) Resumo executivo do Sprint 4

- Tabela de recomenda??es criada por PDV com 1 a 3 a??es por criticidade.
- Recomenda??es associadas ao `driver_principal` detectado no Sprint 3.
- Base principal vinculada aos planos de a??o para uso no dashboard.
